# 01 — QC workbook inspection and standardization audit

Run all cells with SurgeWise's existing `.venv` Python kernel. The notebook reads the
UPRI-NOAH workbook, audits the Quezon City weekly series, and builds an in-memory
`standardized_qc` table for review. The Regional and Multi-Setting sheets are listed
for context. All source and processed files are read-only in this notebook.

See the [source inventory](../data/reference/upri_noah/sources.json),
[upstream README](../data/raw/upri_noah/README.md), and
[existing integration audit](../docs/upri-integration-audit.md) for provenance and limitations.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

start = Path.cwd()
root = next(
    (p for p in (start, *start.parents) if (p / "services/engine/upri_data.py").is_file()),
    None,
)
if root is None:
    raise RuntimeError("Start the kernel from the SurgeWise checkout or its notebooks directory.")

sys.path.insert(0, str(root))
from services.engine.upri_data import AREAS, WORKBOOK, load_weekly, sha256

if not WORKBOOK.is_file():
    raise FileNotFoundError(
        f"Missing {WORKBOOK.relative_to(root)}. See data/reference/upri_noah/sources.json."
    )

manifest = json.loads(
    (root / "data/reference/upri_noah/sources.json").read_text(encoding="utf-8")
)
workbook_path = WORKBOOK.relative_to(root).as_posix()
source = next(entry for entry in manifest["sources"] if entry["path"] == workbook_path)
actual_hash = sha256(WORKBOOK)
if actual_hash != source["sha256"]:
    raise ValueError(f"Workbook SHA-256 differs from the pinned source for {workbook_path}.")

display(pd.Series({
    "Workbook": workbook_path,
    "SHA-256": actual_hash,
    "Source commit": source["sourceCommitId"],
    "Source URL": source["sourceUrl"],
    "License": manifest["license"],
    "Python": sys.executable,
    "pandas": pd.__version__,
}, name="Provenance"))

Workbook         data/raw/upri_noah/Dengue_Environmental Vars_D...
SHA-256          94c65ce24e13c132007f67da9b4d10f7023e9429fff79c...
Source commit             02926278d570db8950da368d44db544ae0097615
Source URL       https://github.com/pelitro-rcw/dengue-environm...
License                               ODC-ODbL-1.0 (source README)
Python           /home/lumpia/Desktop/Workspace/SurgeWise/.venv...
pandas                                                       2.3.3
Name: Provenance, dtype: object

## Workbook inventory and metadata

The embedded dictionary and summary describe the source. The
[upstream validation report](../data/reference/upri_noah/environmental_validation_report.xlsx)
is a separate reference. The repository audit documents differences between
its schema and the pinned workbook.

In [2]:
qc_sheet, case_column, _ = AREAS["QUEZON_CITY"]

with pd.ExcelFile(WORKBOOK, engine="openpyxl") as workbook:
    required_sheets = {"Dataset Summary", "Data Dictionary", qc_sheet}
    absent = required_sheets.difference(workbook.sheet_names)
    if absent:
        raise ValueError(f"Missing required workbook sheets: {sorted(absent)}")
    display(pd.DataFrame({"Worksheet": workbook.sheet_names}))
    raw_qc = workbook.parse(qc_sheet)
    summary = workbook.parse("Dataset Summary", header=None)
    dictionary = workbook.parse("Data Dictionary", header=1)

qc_fields = pd.DataFrame({"CODE": raw_qc.columns, "dtype": raw_qc.dtypes.astype(str).to_numpy()})
dictionary_gaps = qc_fields.loc[~qc_fields["CODE"].isin(dictionary["CODE"]), "CODE"].tolist()
print(f"QC Data: {len(raw_qc):,} rows, {raw_qc.shape[1]} columns")
print(f"QC columns absent from the embedded dictionary: {dictionary_gaps}")
display(qc_fields.merge(dictionary, on="CODE", how="left"))
display(summary.loc[summary[0].eq("Weekly alignment"), [0, 1]])
display(raw_qc.head())

,Worksheet
0,Dataset Summary
1,Data Dictionary
2,QC Data
3,Regional Data
4,Multi-Setting Data


QC Data: 832 rows, 24 columns
QC columns absent from the embedded dictionary: ['RF_PAGASA']


,CODE,dtype,Variable Name,Description,Source Sheet,Data Source,Unit of Measurement
0,YR,int64,Year,Calendar year of observation (2010–2025),All Sheets,Surveillance / NASA / DOH / OpenDengue,Year (integer)
1,WN,int64,Week Number,ISO epidemiological week number (1–52 or 1–53)...,All Sheets,Derived from source dates,Integer (1–53)
2,DC_QC,int64,"Dengue Cases, QC",Weekly confirmed and probable dengue cases rep...,QC Data,QCESD via PIDSR surveillance system,Count (integer)
3,RF_NASA,float64,"Rainfall, NASA IMERG",Weekly total rainfall (mm) from NASA GPM IMERG...,"QC Data, Multi-Setting Data",NASA GES DISC GPM_3IMERGDF v07,mm per week (float)
4,RF_PAGASA,float64,NaN,NaN,NaN,NaN,NaN
5,Temp_ERA5-Land,float64,"Temperature, ERA5-Land (mean)",Weekly mean of daily mean 2 m air temperature ...,"QC Data, Regional Data, Multi-Setting Data",ERA5-Land reanalysis (Copernicus C3S),degrees Celsius (float)
6,TempMax_ERA5-Land,float64,"Temperature, ERA5-Land (daily high, weekly mean)",Weekly mean of daily maximum 2 m air temperatu...,QC Data,ERA5-Land reanalysis via Open-Meteo,degrees Celsius (float)
7,TempMin_ERA5-Land,float64,"Temperature, ERA5-Land (daily low, weekly mean)",Weekly mean of daily minimum 2 m air temperatu...,QC Data,ERA5-Land reanalysis via Open-Meteo,degrees Celsius (float)
8,RH_ERA5-Land,float64,"Relative Humidity, ERA5-Land (mean)",Weekly mean of daily mean 2 m relative humidit...,QC Data,ERA5-Land reanalysis via Open-Meteo,percent (float)
9,RHMax_ERA5-Land,float64,"Relative Humidity, ERA5-Land (daily high, week...",Weekly mean of daily maximum relative humidity...,QC Data,ERA5-Land reanalysis via Open-Meteo,percent (float)


,0,1
38,Weekly alignment,All new columns follow the same YR/WN conventi...


,YR,WN,DC_QC,RF_NASA,RF_PAGASA,Temp_ERA5-Land,TempMax_ERA5-Land,TempMin_ERA5-Land,RH_ERA5-Land,RHMax_ERA5-Land,...,SH_ERA5-Land,Temp_PAGASA,RH_PAGASA,VaporPressureSat_PAGASA,VaporPressureAct_PAGASA,SH_PAGASA,FLAG_COVID,FLAG_SINGLE_CELL_RF,FLAG_PLAUSIBILITY,FLAG_PRESSURE_SH_GAP
0,2010,1,15,5.719467,NaN,25.471429,28.628571,23.128571,77.428571,91.285714,...,15.752594,NaN,NaN,NaN,NaN,NaN,0,1,0,0
1,2010,2,35,16.374799,NaN,24.842857,28.100000,22.485714,75.571429,88.714286,...,14.810863,NaN,NaN,NaN,NaN,NaN,0,1,0,0
2,2010,3,38,8.326888,NaN,24.485714,26.957143,22.685714,76.857143,86.428571,...,14.696193,NaN,NaN,NaN,NaN,NaN,0,1,0,0
3,2010,4,22,9.108687,NaN,24.571429,27.800000,22.357143,73.571429,87.428571,...,14.112502,NaN,NaN,NaN,NaN,NaN,0,1,0,0
4,2010,5,30,1.393803,NaN,25.385714,29.614286,22.014286,75.000000,94.285714,...,15.136500,NaN,NaN,NaN,NaN,NaN,0,1,0,0


## Source keys, coverage, and standardization

The dictionary describes ISO weeks. The summary describes source blocks with
rare 53rd blocks folded into block 52. Following the current project loader,
`source_block = YR * 52 + WN - 1` is an ordinal. Exact calendar dates remain unresolved.

The loader rejects duplicate keys, invalid year or week values, invalid
case counts, and nonbinary flags. It sorts the series and leaves absent
source blocks as null rows. Observed rows retain the original columns, units,
counts, flags, and nulls. Missing blocks are audited across the workbook's
observed year span, including the edges of the first and last years.

In [3]:
standardized_qc, case_column, _ = load_weekly("QUEZON_CITY", workbook=WORKBOOK)
standardized_qc.index.name = "source_block"

observed_blocks = standardized_qc.index[standardized_qc[case_column].notna()]
year_counts = raw_qc.groupby("YR")["WN"].nunique()
years = range(int(year_counts.index.min()), int(year_counts.index.max()) + 1)
expected_blocks = pd.RangeIndex(years.start * 52, years.stop * 52, name="source_block")
missing_blocks = expected_blocks.difference(observed_blocks)
coverage = year_counts.reindex(years, fill_value=0).rename("Observed blocks").to_frame()
coverage["Missing blocks"] = 52 - coverage["Observed blocks"]

display(coverage)
print(f"Missing blocks within the observed year span: {len(missing_blocks):,}")
if len(missing_blocks):
    display(pd.DataFrame({"YR": missing_blocks // 52, "WN": missing_blocks % 52 + 1},
                         index=missing_blocks))

observed_qc = standardized_qc.loc[observed_blocks]
pd.testing.assert_frame_equal(
    observed_qc.reset_index(drop=True),
    raw_qc.sort_values(["YR", "WN"]).reset_index(drop=True),
    check_dtype=False,
    check_exact=True,
)

,Observed blocks,Missing blocks
YR,,
2010,52,0
2011,52,0
2012,52,0
2013,52,0
2014,52,0
2015,52,0
2016,52,0
2017,52,0
2018,52,0


Missing blocks within the observed year span: 0


## Field coverage and quality flags

Missing-value rates use the observed source rows as the denominator.
The domain checks screen rainfall for negative values and relative humidity
for values outside 0–100%. The flag checks compare COVID-era coverage and the
paired ERA5-Land pressure and specific-humidity blank pattern. Flagged rows
remain available for review, including the COVID-period observations.

In [4]:
coverage_by_field = pd.DataFrame({
    "Missing": observed_qc.isna().sum(),
    "Present": observed_qc.notna().sum(),
    "Missing %": observed_qc.isna().mean().mul(100).round(2),
})
display(coverage_by_field)

flags = [column for column in raw_qc if column.startswith("FLAG_")]
flag_counts = observed_qc[flags].apply(pd.Series.value_counts).reindex([0, 1]).fillna(0)
display(flag_counts.T.astype(int).rename(columns={0: "Clear", 1: "Flagged"}))

weather_review = []
for field in raw_qc:
    if not field.startswith(("RF_", "RH")):
        continue
    values = pd.to_numeric(observed_qc[field], errors="raise").dropna()
    outside = ~np.isfinite(values) | values.lt(0)
    if field.startswith("RH"):
        outside |= values.gt(100)
    weather_review.append({"Field": field, "Checked": len(values), "Outside domain": int(outside.sum())})
display(pd.DataFrame(weather_review))

covid_disagreement = observed_qc["FLAG_COVID"].eq(1).ne(observed_qc["YR"].isin([2020, 2021]))
pressure_gap = observed_qc["FLAG_PRESSURE_SH_GAP"].eq(1)
pressure_disagreement = (
    pressure_gap.ne(observed_qc["Pressure_ERA5-Land"].isna())
    | pressure_gap.ne(observed_qc["SH_ERA5-Land"].isna())
)
print(f"COVID flag disagreements: {int(covid_disagreement.sum())}")
print(f"Pressure/SH gap flag disagreements: {int(pressure_disagreement.sum())}")
display(standardized_qc.head())

,Missing,Present,Missing %
YR,0,832,0.00
WN,0,832,0.00
DC_QC,0,832,0.00
RF_NASA,0,832,0.00
RF_PAGASA,572,260,68.75
Temp_ERA5-Land,0,832,0.00
TempMax_ERA5-Land,0,832,0.00
TempMin_ERA5-Land,0,832,0.00
RH_ERA5-Land,0,832,0.00
RHMax_ERA5-Land,0,832,0.00


,Clear,Flagged
FLAG_COVID,728,104
FLAG_SINGLE_CELL_RF,0,832
FLAG_PLAUSIBILITY,832,0
FLAG_PRESSURE_SH_GAP,819,13


,Field,Checked,Outside domain
0,RF_NASA,832,0
1,RF_PAGASA,260,0
2,RH_ERA5-Land,832,0
3,RHMax_ERA5-Land,832,0
4,RHMin_ERA5-Land,832,0
5,RH_PAGASA,352,0


COVID flag disagreements: 0
Pressure/SH gap flag disagreements: 0


,YR,WN,DC_QC,RF_NASA,RF_PAGASA,Temp_ERA5-Land,TempMax_ERA5-Land,TempMin_ERA5-Land,RH_ERA5-Land,RHMax_ERA5-Land,...,SH_ERA5-Land,Temp_PAGASA,RH_PAGASA,VaporPressureSat_PAGASA,VaporPressureAct_PAGASA,SH_PAGASA,FLAG_COVID,FLAG_SINGLE_CELL_RF,FLAG_PLAUSIBILITY,FLAG_PRESSURE_SH_GAP
source_block,,,,,,,,,,,,,,,,,,,,,
104520,2010,1,15,5.719467,NaN,25.471429,28.628571,23.128571,77.428571,91.285714,...,15.752594,NaN,NaN,NaN,NaN,NaN,0,1,0,0
104521,2010,2,35,16.374799,NaN,24.842857,28.100000,22.485714,75.571429,88.714286,...,14.810863,NaN,NaN,NaN,NaN,NaN,0,1,0,0
104522,2010,3,38,8.326888,NaN,24.485714,26.957143,22.685714,76.857143,86.428571,...,14.696193,NaN,NaN,NaN,NaN,NaN,0,1,0,0
104523,2010,4,22,9.108687,NaN,24.571429,27.800000,22.357143,73.571429,87.428571,...,14.112502,NaN,NaN,NaN,NaN,NaN,0,1,0,0
104524,2010,5,30,1.393803,NaN,25.385714,29.614286,22.014286,75.000000,94.285714,...,15.136500,NaN,NaN,NaN,NaN,NaN,0,1,0,0


In [5]:
import json

from IPython.display import display
from services.engine.domain.spatial import annual_weights
from services.engine.upri_data import load_annual

# Build a table with one row per barangay and year. Blank counts stay missing.
annual_wide = load_annual()
count_columns = annual_wide.filter(regex=r"^DC_\d{4}$").columns

annual_barangay = annual_wide.melt(
    id_vars=["ADM4_PCODE", "Barangay"],
    value_vars=count_columns,
    var_name="source_field",
    value_name="cases",
)
annual_barangay["year"] = annual_barangay.pop("source_field").str[3:].astype(int)
annual_barangay["cases"] = annual_barangay["cases"].astype("Int64")
annual_barangay = annual_barangay[
    ["ADM4_PCODE", "Barangay", "year", "cases"]
].sort_values(["year", "ADM4_PCODE"], ignore_index=True)

# Count reported and missing values for each year.
annual_coverage = annual_barangay.groupby("year").agg(
    total_barangays=("ADM4_PCODE", "size"),
    reported_counts=("cases", "count"),
)
annual_coverage["missing_counts"] = (
    annual_coverage["total_barangays"] - annual_coverage["reported_counts"]
)
annual_coverage["complete"] = annual_coverage["missing_counts"].eq(0)

print("Annual barangay coverage")
display(annual_coverage)

# Check which complete year the existing allocation baseline selects.
records = json.loads(annual_wide.to_json(orient="records"))
selected_years = {}

for issue_year in (2024, 2025, 2026):
    weight_year, _ = annual_weights(records, issue_year)
    selected_years[issue_year] = weight_year

print("Annual weight year selected by forecast issue year")
for issue_year, weight_year in selected_years.items():
    print(f"{issue_year} → {weight_year}")

Annual barangay coverage


,total_barangays,reported_counts,missing_counts,complete
year,,,,
2010,142,139,3,False
2011,142,140,2,False
2012,142,139,3,False
2013,142,141,1,False
2014,142,130,12,False
2015,142,140,2,False
2016,142,140,2,False
2017,142,142,0,True
2018,142,141,1,False


Annual weight year selected by forecast issue year
2024 → 2019
2025 → 2019
2026 → 2024


## Handoff to exploratory analysis

Review coverage, out-of-domain counts, flag disagreements, and dictionary
gaps before proceeding to [02 — Exploratory data analysis](02_eda.ipynb).
Recreate `standardized_qc` by running this notebook against the pinned workbook.
Keep the source-block convention and inspect any missing blocks before applying
positional lags. Environmental publication timing and historical data vintages
still need evidence before treating any field as available at a forecast origin.

## Annual population and weekly log incidence

This prepares the target used by Carvajal et al. for the QC Random Forest replication. Join each observed weekly count to a citywide denominator for its source year. The denominator table uses official PSA census counts on their census reference dates. Between censuses, estimate population as of July 1 using constant compounded growth within each census interval. For July 1, 2025, extend the growth rate from the 2020 and 2024 counts. This extrapolation is a project estimate. An older-base PSA projection is available as a separate reference.

The weekly target is the natural log of reported cases per 1,000 residents. Each census-year denominator uses that year's census reference date. Intermediate and projected years have July 1 reference dates. The table is for retrospective analysis and preserves the original weekly counts. Run the source-key and coverage cell above first.

[PSA Regional Social and Economic Trends, Table 1.1](https://rssoncr.psa.gov.ph/system/files/publication/2026%20Regional%20Social%20Economic%20Trends%20%28RSET%29.pdf) · [PSA 2020-based projection, Table 4](https://psa.gov.ph/system/files/dhsd/Press%20Release%20%282020%20CBPP%20Subnational%29.pdf) · [Carvajal et al., population method](https://link.springer.com/article/10.1186/s12879-018-3066-0)


In [6]:
import numpy as np
import pandas as pd
from IPython.display import display

if "standardized_qc" not in globals() or "case_column" not in globals():
    raise RuntimeError("Run the source keys, coverage, and standardization cell first.")

census = pd.Series(
    [2_761_720, 2_936_116, 2_960_048, 3_084_270],
    index=pd.to_datetime(["2010-05-01", "2015-08-01", "2020-05-01", "2024-07-01"]),
    name="population",
)
census_dates = {date.year: date for date in census.index}
population_rows = []

for year in range(2010, 2026):
    if year in census_dates:
        reference_date = census_dates[year]
        population = int(census.loc[reference_date])
        basis = "PSA census"
    else:
        reference_date = pd.Timestamp(year=year, month=7, day=1)
        if reference_date > census.index[-1]:
            start, end = census.index[-2:]
            basis = "Extrapolated from 2020–2024"
        else:
            right = census.index.searchsorted(reference_date)
            start, end = census.index[right - 1:right + 1]
            basis = f"Interpolated between {start.year} and {end.year}"

        share_of_interval = (reference_date - start) / (end - start)
        start_count, end_count = census.loc[start], census.loc[end]
        population = round(start_count * (end_count / start_count) ** share_of_interval)

    population_rows.append({
        "year": year,
        "reference_date": reference_date.date(),
        "population": population,
        "basis": basis,
    })

population_annual = pd.DataFrame(population_rows).set_index("year")
qc_incidence = standardized_qc.copy()
qc_incidence["qc_population"] = qc_incidence["YR"].map(population_annual["population"])
observed_cases = qc_incidence[case_column].notna()
if qc_incidence.loc[observed_cases, "qc_population"].isna().any():
    raise ValueError("Missing QC population denominator for an observed year.")
if not qc_incidence.loc[observed_cases, case_column].gt(0).all():
    raise ValueError("Log incidence requires positive observed case counts.")

qc_incidence["cases_per_1000"] = qc_incidence[case_column] / qc_incidence["qc_population"] * 1_000
qc_incidence["log_cases_per_1000"] = np.log(qc_incidence["cases_per_1000"])
display(population_annual)
display(qc_incidence[["YR", "WN", case_column, "qc_population",
                      "cases_per_1000", "log_cases_per_1000"]].head())


,reference_date,population,basis
year,,,
2010,2010-05-01,2761720,PSA census
2011,2011-07-01,2799537,Interpolated between 2010 and 2015
2012,2012-07-01,2832441,Interpolated between 2010 and 2015
2013,2013-07-01,2865641,Interpolated between 2010 and 2015
2014,2014-07-01,2899229,Interpolated between 2010 and 2015
2015,2015-08-01,2936116,PSA census
2016,2016-07-01,2940722,Interpolated between 2015 and 2020
2017,2017-07-01,2945748,Interpolated between 2015 and 2020
2018,2018-07-01,2950783,Interpolated between 2015 and 2020


,YR,WN,DC_QC,qc_population,cases_per_1000,log_cases_per_1000
source_block,,,,,,
104520,2010,1,15,2761720,0.005431,-5.215559
104521,2010,2,35,2761720,0.012673,-4.368261
104522,2010,3,38,2761720,0.013760,-4.286023
104523,2010,4,22,2761720,0.007966,-4.832566
104524,2010,5,30,2761720,0.010863,-4.522412


## Flood advisory archive pilot

The paper reconstructed a severe Metro Manila flood indicator from dated MMDA advisories and news cross-checks. This pilot looks for equivalent evidence of flooding and road impacts inside Quezon City. It samples three known event windows from different years and a comparison window used to check whether public search can establish a monitored non-event week. The windows below are calendar dates in Philippine time. Correspondence with the workbook's source blocks remains unresolved.

| Calendar window | Evidence recovered | What can be coded today |
|---|---|---|
| 2012-08-06 to 2012-08-12 | The 7 August NDRRMC road bulletin lists QC roads affected by flooding in Roxas, Damayan, and other locations. Some roads were impassable to all vehicles. [NDRRMC SitRep 2, road status](https://ndrrmc.gov.ph/attachments/article/2051/SitRep_No_02_re_Effects_of_Southwest_Monsoon_Enhanced_by_Tropical_Storm_HAIKUI_as_of_7AUG2012_1700H.pdf#page=7) | Documented QC flood with road restrictions. Citywide road coverage unknown. |
| 2016-02-01 to 2016-02-07 | A targeted public search did not locate a qualifying QC advisory or documentation of complete monitoring for this period. | Unknown. |
| 2024-10-21 to 2024-10-27 | The QC calamity resolution documents flooding and impassable roads during Kristine. A 25 October QC update also describes flood-related drainage work. [QC resolution](https://quezoncity.gov.ph/wp-content/uploads/2024/10/Resolution-No.-SP-9832-S-2024.pdf) · [25 October update](https://quezoncity.gov.ph/declogging-operations-by-qc-engineering-department-6/) | Documented QC flood with road restrictions. Citywide road coverage unknown. |
| 2025-08-25 to 2025-08-31 | QC published timed road-depth updates on 30 August. PAGASA corroborated the event on 1 September. The dated reports refer to the same flood episode. [QC road advisory](https://quezoncity.gov.ph/status-of-roads-in-quezon-city-0305-p-m/) · [PAGASA event report](https://www.pagasa.dost.gov.ph/press-release/186?page=15) | Documented QC flood. Citywide road coverage unknown. |

The pilot recovered positive flood evidence in three windows and established no verified zero weeks. The records report individual affected roads and areas. We still need a QC severity rule and evidence of which roads and weeks were routinely monitored before assigning the paper-style binary flag. An event archive should preserve event time, posting time, road or barangay, vehicle passability, and the original source link. Repeated updates about a single episode should be linked to that event.

A dated city monitoring export is the next source to verify for consistent coverage. [Carvajal et al., flood definition](https://link.springer.com/article/10.1186/s12879-018-3066-0) · [QC disaster management data request](https://quezoncity.gov.ph/departments/quezon-city-disaster-and-risk-reduction-management-office/) · [MMDA example flood-record request](https://www.foi.gov.ph/agencies/mmda/the-list-of-floods-in-metro-manila-from-2010-to-2024/)
